# Reflectance or display color?

Distinguish stored values, reflectance and display settings.

**Task** · Change the display range of the same Landsat image and compare it with mean reflectance.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
raw_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi).filterDate('2020-05-01', '2020-10-01'))
def landsat_sr(image):
    clear = image.select('QA_PIXEL').bitwiseAnd(31).eq(0)
    unsaturated = image.select('QA_RADSAT').eq(0)
    return image.select(['SR_B2','SR_B3','SR_B4','SR_B5']).multiply(0.0000275).add(-0.2).updateMask(clear.And(unsaturated))
sr = raw_collection.map(landsat_sr).median().clip(roi)
means = sr.reduceRegion(ee.Reducer.mean(), roi, 30, maxPixels=2_000_000)

Apply a scale factor of 0.0000275 and an offset of −0.2 to Collection 2 stored values. Exclude QA_PIXEL fill, dilated-cloud, cirrus, cloud and shadow flags, and saturated pixels. RGB = B4, B3, B2.

In [ ]:
print('images:', raw_collection.size().getInfo())
print('mean surface reflectance:', means.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(sr, {'bands':['SR_B4','SR_B3','SR_B2'], 'min':0, 'max':0.3}, 'RGB max 0.3')
Map.addLayer(sr, {'bands':['SR_B4','SR_B3','SR_B2'], 'min':0, 'max':0.6}, 'RGB max 0.6')
Map

Toggle the two layers. Explain why changing display brightness does not change the calculated mean reflectance. Describe the change needed to compare this result with an unmasked composite.

## Data and interpretation

- [Landsat 8 Collection 2 Level 2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.